# 2D slice visualization for `circuit_test`

Default plane is **x–z** (thin-y domain: `amr.n_cell = 256 4 256`).

In [ ]:
from pathlib import Path
import os
import shutil

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mplconfig")

import numpy as np
import yt
import logging
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from tqdm import tqdm

# PACE module path if ffmpeg is not already on PATH
_ffmpeg = shutil.which("ffmpeg") or (
    "/usr/local/pace-apps/spack/packages/linux-rhel9-x86_64_v3/"
    "gcc-12.3.0/ffmpeg-7.1-jm6ru4wgcqsptdq2ihjgsowowk3rq5az/bin/ffmpeg"
)
mpl.rcParams["animation.ffmpeg_path"] = _ffmpeg

yt.set_log_level("ERROR")
logging.getLogger("yt").setLevel(logging.ERROR)

diag_dir = Path("diags")
plotfiles = sorted(
    diag_dir.glob("plt*"),
    key=lambda path: int(path.name.removeprefix("plt")),
)
assert plotfiles, f"No plotfiles found in {diag_dir.resolve()}"

plotfile = plotfiles[-1]
ds = yt.load(str(plotfile))
grid = ds.covering_grid(
    level=0,
    left_edge=ds.domain_left_edge,
    dims=ds.domain_dimensions,
)


def load_fields(covering_grid):
    Ex = covering_grid[("boxlib", "Ex")].to_ndarray()
    Ey = covering_grid[("boxlib", "Ey")].to_ndarray()
    Ez = covering_grid[("boxlib", "Ez")].to_ndarray()
    Bx = covering_grid[("boxlib", "Bx")].to_ndarray()
    By = covering_grid[("boxlib", "By")].to_ndarray()
    Bz = covering_grid[("boxlib", "Bz")].to_ndarray()
    fields = {
        "Ex": Ex,
        "Ey": Ey,
        "Ez": Ez,
        "|E|": np.sqrt(Ex**2 + Ey**2 + Ez**2),
        "Bx": Bx,
        "By": By,
        "Bz": Bz,
        "|B|": np.sqrt(Bx**2 + By**2 + Bz**2),
    }
    try:
        fields["epsilon"] = covering_grid[("boxlib", "epsilon")].to_ndarray()
    except Exception:
        pass
    return fields


fields = load_fields(grid)

print(f"Loaded {plotfile}")
print(f"Available fields: {', '.join(fields)}")
print(f"Grid shape: {next(iter(fields.values())).shape}  # x, y, z")
print(f"Domain lo/hi: {ds.domain_left_edge} / {ds.domain_right_edge}")

# --- selection ---
variable = "|E|"  # Ex, Ey, Ez, |E|, Bx, By, Bz, |B|, epsilon
plane = "xy"      # xy, xz, yz  (normal axis is the missing one)
slice_index = None  # None -> center of the normal axis
# -----------------

axis_name = "xyz"
ax0, ax1 = {"xy": (0, 1), "xz": (0, 2), "yz": (1, 2)}[plane]
normal = {"xy": 2, "xz": 1, "yz": 0}[plane]

data = fields[variable]
dims = data.shape
centers = [n // 2 for n in dims]
fixed_index = centers[normal] if slice_index is None else int(slice_index)

idx = [slice(None)] * 3
idx[normal] = fixed_index
slice2d = np.asarray(data[tuple(idx)])


def axis_centers(axis):
    edges = np.linspace(
        float(ds.domain_left_edge[axis]),
        float(ds.domain_right_edge[axis]),
        dims[axis] + 1,
    )
    return 0.5 * (edges[:-1] + edges[1:])


c0 = axis_centers(ax0)
c1 = axis_centers(ax1)
extent = [
    float(c0[0] * 1e3),
    float(c0[-1] * 1e3),
    float(c1[0] * 1e3),
    float(c1[-1] * 1e3),
]

signed = not variable.startswith("|") and variable != "epsilon"
vmax = float(np.max(np.abs(slice2d))) if signed else float(slice2d.max())
vmin = -vmax if signed else float(slice2d.min())
cmap = "RdBu_r" if signed else "viridis"

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(
    slice2d.T,
    origin="lower",
    extent=extent,
    aspect="auto",
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
)
cb = fig.colorbar(im, ax=ax, shrink=0.85)
cb.set_label(variable)
ax.set_xlabel(f"{axis_name[ax0]} (mm)")
ax.set_ylabel(f"{axis_name[ax1]} (mm)")
ax.set_title(
    f"{variable} on {plane} plane at {axis_name[normal]}-index {fixed_index}"
)
fig.tight_layout()
plt.show()

In [ ]:
# --- selection ---
variable = "|E|"  # Ex, Ey, Ez, |E|, Bx, By, Bz, |B|, epsilon
plane = "xy"      # xy, xz, yz
slice_index = None  # None -> center of the normal axis
stride = 1
n_workers = 4  # parallel plotfile readers; 1 = serial
# -----------------

axis_name = "xyz"
ax0, ax1 = {"xy": (0, 1), "xz": (0, 2), "yz": (1, 2)}[plane]
normal = {"xy": 2, "xz": 1, "yz": 0}[plane]
frames = plotfiles[::stride]


def load_variable(grid, name):
    """Load only the field(s) needed for `name` (not all EM components)."""
    if name in ("Ex", "Ey", "Ez", "Bx", "By", "Bz", "epsilon", "mu"):
        return grid[("boxlib", name)].to_ndarray()
    if name in ("|E|", "|B|"):
        prefix = name[1]
        comps = [grid[("boxlib", f"{prefix}{ax}")].to_ndarray() for ax in "xyz"]
        return np.sqrt(comps[0] ** 2 + comps[1] ** 2 + comps[2] ** 2)
    raise KeyError(name)


def load_slice(path):
    # Note: these are AMReX plotfiles (via yt), not .npy. yt still has to read
    # full 3D boxes that intersect the slice; the big win is avoiding unused fields.
    frame_ds = yt.load(str(path))
    frame_grid = frame_ds.covering_grid(
        level=0,
        left_edge=frame_ds.domain_left_edge,
        dims=frame_ds.domain_dimensions,
    )
    field = load_variable(frame_grid, variable)
    dims = field.shape
    fixed = dims[normal] // 2 if slice_index is None else int(slice_index)

    idx = [slice(None)] * 3
    idx[normal] = fixed
    slice2d = np.asarray(field[tuple(idx)])

    def centers_1d(axis):
        edges = np.linspace(
            float(frame_ds.domain_left_edge[axis]),
            float(frame_ds.domain_right_edge[axis]),
            dims[axis] + 1,
        )
        return 0.5 * (edges[:-1] + edges[1:])

    return (
        float(frame_ds.current_time),
        centers_1d(ax0),
        centers_1d(ax1),
        slice2d,
        fixed,
    )


if n_workers <= 1:
    results = [load_slice(path) for path in tqdm(frames)]
else:
    # Threads are notebook-safe (no pickle). For ~5-10x more, run as a .py
    # script with ProcessPoolExecutor instead.
    from concurrent.futures import ThreadPoolExecutor

    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        results = list(tqdm(pool.map(load_slice, frames), total=len(frames)))

times = [r[0] for r in results]
c0, c1 = results[0][1], results[0][2]
slices = [r[3] for r in results]
fixed_index = results[0][4]

times = np.asarray(times)
slices = np.asarray(slices)

extent = [
    float(c0[0] * 1e3),
    float(c0[-1] * 1e3),
    float(c1[0] * 1e3),
    float(c1[-1] * 1e3),
]

signed = not variable.startswith("|") and variable != "epsilon"
cmap = "RdBu_r" if signed else "viridis"


def frame_clim(frame):
    if signed:
        vmax = float(np.max(np.abs(frame)))
        vmax = max(vmax, np.finfo(float).eps)
        return -vmax, vmax
    vmin = float(frame.min())
    vmax = float(frame.max())
    if vmin == vmax:
        vmax = vmin + np.finfo(float).eps
    return vmin, vmax


vmin0, vmax0 = frame_clim(slices[0])

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(
    slices[0].T,
    origin="lower",
    extent=extent,
    aspect="auto",
    cmap=cmap,
    vmin=vmin0,
    vmax=vmax0,
)
cb = fig.colorbar(im, ax=ax, shrink=0.85)
cb.set_label(variable)
ax.set_xlabel(f"{axis_name[ax0]} (mm)")
ax.set_ylabel(f"{axis_name[ax1]} (mm)")
title = ax.set_title(
    f"{variable} on {plane} at {axis_name[normal]}-index {fixed_index}, "
    f"t = {times[0]:.3e} s"
)
fig.tight_layout()


def update(frame_index):
    frame = slices[frame_index]
    im.set_data(frame.T)
    im.set_clim(*frame_clim(frame))
    title.set_text(
        f"{variable} on {plane} at {axis_name[normal]}-index {fixed_index}, "
        f"t = {times[frame_index]:.3e} s"
    )
    return im, title


animation = FuncAnimation(fig, update, frames=len(slices), interval=50, blit=False)

safe_var = variable.replace("|", "").replace(" ", "")
out_dir = Path("circuit_test_run")
out_dir.mkdir(parents=True, exist_ok=True)
video_path = out_dir / f"{safe_var}_{plane}.mp4"
writer = FFMpegWriter(fps=20, codec="mpeg4", bitrate=2400)
animation.save(video_path, writer=writer, dpi=150)
plt.close(fig)

print(f"Wrote {video_path}")
print(f"Frames: {len(slices)}")
